#### This notebook demonstrates the full flow of STAC specification generation, starting from item (for a raster and a vector layer) as well as the corresponding catalog and collection specifications. 

- Input Data (Layer data, Layer style file, Descriptions): available locally 
- Output : saved locally

1. Imports, constants, paths

In [89]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

import rasterio
import os

import json
import xml.etree.ElementTree as ET
import datetime
# from datetime import datetime, timezone

from rasterio.warp import transform_bounds

from matplotlib.colors import ListedColormap, Normalize
from shapely.geometry import mapping, box, Polygon

import sys
sys.path.append('..')
import constants

import pystac
from pystac.extensions.table import TableExtension
from pystac import Asset, MediaType
from pystac.extensions.classification import ClassificationExtension, Classification
from pystac.extensions.raster import RasterExtension,RasterBand
from pystac.extensions.projection import ProjectionExtension

In [90]:
data_dir = "../data/"
corestack_dir = os.path.join(data_dir, 'CorestackCatalogs')

output_dir = os.path.join(data_dir,'STAC_output')

2. Map layernames with style file names

In [91]:
layernames_stylefiles_df = pd.DataFrame({
    'layer_name' : ['admin_boundary_vector',
                    'mgnrega_vector',
                    'lulc_raster',
                    'terrain_raster',
                    'terrain_vector',
                    'clart_raster',
                    'swb_vector',
                    'drainage_lines_vector',
                    'cropping_intensity_vector',
                    'prec_annual_vector',
                    'tree_cover_ccd_raster',
                    'tree_cover_ch_raster',
                    'tree_cover_change_raster',
                    'aquifer_vector',
                    'soge_vector',
                    'restoration_raster',
                    'change_detection_tree_cover_gain_raster',
                    'change_detection_cropping_intensity_raster',
                    'change_detection_tree_cover_loss_raster',
                    'change_detection_cropping_reduction_raster',
                    'change_detection_urbanization_raster',
                    'drought_frequency_vector',
                    'runoff_annual_vector',
                    'well_depth_annual_vector',
                    'deltaG_annual_vector',
                    'deltaG_fortnightly_vector'],
    'style_file_name' : [
        'Administrative-Boundary-Style.qml',
        'NREG-Assets-Classified-Style.qml',
        'LULC0_12class.qml',
        'terrain_1-12class.qml',
        'Terrain-Vector-Layer-Style.qml',
        'CLART-Layer-Style.qml',
        'Surface-Waterbody-style.qml',
        'Drainage-Layer-Style.qml',
        'Cropping_intensity.qml',
        'Precipitation_Style.qml',
        'ccd_style.qml',
        'tree_cover_change.qml',
        'tree_cover_change.qml',
        'Aquifer_style.qml',
        'SOGE_style.qml',
        'Restoration_style.qml',
        'tree_cover_gain.qml',
        'Cropping_Intensity_climate_change.qml',
        'tree_cover_loss.qml',
        'cropping_reduction.qml',
        'Urbanization_climate_change.qml',
        'Drought_style.qml',
        'Runoff_style.qml',
        'MWS-Well-Depth-18_23.qml',
        'MWS-Well-Depth-18_23.qml',
        'MWS-Well-Depth-18_23.qml'
    ]
})

In [92]:
layernames_stylefiles_df

,layer_name,style_file_name
0,admin_boundary_vector,Administrative-Boundary-Style.qml
1,mgnrega_vector,NREG-Assets-Classified-Style.qml
2,lulc_raster,LULC0_12class.qml
3,terrain_raster,terrain_1-12class.qml
4,terrain_vector,Terrain-Vector-Layer-Style.qml
5,clart_raster,CLART-Layer-Style.qml
6,swb_vector,Surface-Waterbody-style.qml
7,drainage_lines_vector,Drainage-Layer-Style.qml
8,cropping_intensity_vector,Cropping_intensity.qml
9,prec_annual_vector,Precipitation_Style.qml


3. State - district - block mapping (will come from DB later on)

In [93]:
block_district_state_df = pd.DataFrame({
    'block' : ['gobindpur','mirzapur','koraput','badlapur'],
    'district' : ['saraikela-kharsawan','mirzapur','koraput','jaunpur'],
    'state' : ['jharkhand','uttar_pradesh','odisha','uttar_pradesh']
})

block_district_state_df

,block,district,state
0,gobindpur,saraikela-kharsawan,jharkhand
1,mirzapur,mirzapur,uttar_pradesh
2,koraput,koraput,odisha
3,badlapur,jaunpur,uttar_pradesh


4. Specify data layer of interest

In [94]:
raster_layername = 'lulc_raster'
# raster_layername = 'terrain_raster'

In [95]:
raster_filename = 'jaunpur_badlapur_2019-07-01_2020-06-30_LULCmap_10m.tif'
# raster_filename = 'terrain_raster_jaunpur_badlapur.tif'

In [96]:
raster_path = os.path.join(data_dir,raster_filename)
raster_path

'../data/jaunpur_badlapur_2019-07-01_2020-06-30_LULCmap_10m.tif'

In [97]:
# layer_mapping_df.columns

In [98]:
layernames_stylefiles_df

,layer_name,style_file_name
0,admin_boundary_vector,Administrative-Boundary-Style.qml
1,mgnrega_vector,NREG-Assets-Classified-Style.qml
2,lulc_raster,LULC0_12class.qml
3,terrain_raster,terrain_1-12class.qml
4,terrain_vector,Terrain-Vector-Layer-Style.qml
5,clart_raster,CLART-Layer-Style.qml
6,swb_vector,Surface-Waterbody-style.qml
7,drainage_lines_vector,Drainage-Layer-Style.qml
8,cropping_intensity_vector,Cropping_intensity.qml
9,prec_annual_vector,Precipitation_Style.qml


In [99]:
raster_stylename = layernames_stylefiles_df[layernames_stylefiles_df['layer_name'] == raster_layername]['style_file_name'].iloc[0]
raster_stylename

'LULC0_12class.qml'

In [100]:
raster_style_path = os.path.join(data_dir,raster_stylename)
raster_style_path

'../data/LULC0_12class.qml'

In [101]:
raster_thumbnail_path = os.path.join(
    output_dir,
    raster_filename.split('.')[0] + '_thumbnail.png'
) #TODO: change

In [102]:
# vector_filename = 'admin_boundary_jaunpur_badlapur.geojson'
vector_filename = 'swb3_jaunpur_badlapur.geojson'

In [103]:
vector_path = os.path.join(data_dir,vector_filename)
vector_path

'../data/swb3_jaunpur_badlapur.geojson'

In [104]:
layernames_stylefiles_df['layer_name'].unique()

array(['admin_boundary_vector', 'mgnrega_vector', 'lulc_raster',
       'terrain_raster', 'terrain_vector', 'clart_raster', 'swb_vector',
       'drainage_lines_vector', 'cropping_intensity_vector',
       'prec_annual_vector', 'tree_cover_ccd_raster',
       'tree_cover_ch_raster', 'tree_cover_change_raster',
       'aquifer_vector', 'soge_vector', 'restoration_raster',
       'change_detection_tree_cover_gain_raster',
       'change_detection_cropping_intensity_raster',
       'change_detection_tree_cover_loss_raster',
       'change_detection_cropping_reduction_raster',
       'change_detection_urbanization_raster', 'drought_frequency_vector',
       'runoff_annual_vector', 'well_depth_annual_vector',
       'deltaG_annual_vector', 'deltaG_fortnightly_vector'], dtype=object)

In [105]:
# vector_layername = 'admin_boundary_vector'
vector_layername = 'swb_vector'

In [106]:
vector_stylename = layernames_stylefiles_df[layernames_stylefiles_df['layer_name'] == vector_layername]['style_file_name'].iloc[0]
vector_stylename

'Surface-Waterbody-style.qml'

In [107]:
vector_style_path = os.path.join(data_dir,vector_stylename)
vector_style_path

'../data/Surface-Waterbody-style.qml'

In [108]:
# vector_style_path ='../data/Administrative-Boundary-Style.qml'

In [109]:
vector_thumbnail_path = os.path.join(
    output_dir,
    vector_filename.split('.')[0] + '_thumbnail.png'
)
vector_thumbnail_path

'../data/STAC_output/swb3_jaunpur_badlapur_thumbnail.png'

5. Read column descriptions data (if vector data)

In [110]:
vector_desc_df = pd.read_csv('../data/column_descriptions.csv')

In [111]:
vector_desc_df.rename(columns={'column_name_description' :  'column_description'},inplace=True)

In [112]:
vector_desc_df['layer_name'].unique()

array(['admin_boundary', 'aquifer_vector', 'drainage_lines', 'swb3',
       'swb2', 'change_vector_Afforestation',
       'change_vector_CropIntensity', 'change_vector_Deforestation',
       'change_vector_Degradation', 'change_vector_Urbanization',
       'cropping_intensity_2017-23', 'drought_2017_2022',
       'restoration_vector', 'soge_vector', 'lulc_vector',
       'lulcXplains_clusters', 'terrain_clusters',
       'tree_health_ccd_vector_2017', 'tree_health_overall_change_vector',
       'well_depth_annual', 'well_depth_net_value'], dtype=object)

In [113]:
vector_layername

'swb_vector'

In [114]:
vector_desc_df

,Sno.,layer_name,dataset_name (from DB),column_name,column_description,comments
0,0,admin_boundary,Admin Boundary,ASSET_2001,household assets 2001,NaN
1,1,admin_boundary,Admin Boundary,TOT_M,total male population,NaN
2,2,admin_boundary,Admin Boundary,dist_cen,district census code,NaN
3,3,admin_boundary,Admin Boundary,TOT_F,total female population,NaN
4,4,admin_boundary,Admin Boundary,ADI_2019,Aggregate Development Index (2019),NaN
...,...,...,...,...,...,...
729,729,well_depth_net_value,Hydrology,2017_2018,NaN,NaN
730,730,well_depth_net_value,Hydrology,area_in_ha,NaN,NaN
731,731,well_depth_net_value,Hydrology,id,id,NaN
732,732,well_depth_net_value,Hydrology,2021_2022,NaN,NaN


In [115]:
if ('swb' in vector_layername):
    vector_desc_filtered_df = vector_desc_df[vector_desc_df['layer_name'].str.contains('swb')]
else:
    vector_desc_filtered_df = vector_desc_df[vector_desc_df['layer_name'] == vector_layername.replace('_vector','')] 

4. Read raster data

In [116]:
os.path.relpath(raster_path)

'../data/jaunpur_badlapur_2019-07-01_2020-06-30_LULCmap_10m.tif'

In [117]:
# def get_ground_sample_distance() 
# if proj_epsg != 32644:
#             reprojected_bounds = transform_bounds(src.crs, 'EPSG:32644', *bounds)
#             bbox = list(reprojected_bounds)
#             gsd_x = (reprojected_bounds[2] - reprojected_bounds[0]) / width
#             gsd_y = (reprojected_bounds[3] - reprojected_bounds[1]) / height
#             gsd = (gsd_x + gsd_y) / 2
#         else:
#             bbox = [bounds.left, bounds.bottom, bounds.right, bounds.top]
#             gsd = src.res[0]
            
#     print(f"Raster resolution (GSD): {gsd} meters")

In [118]:
def read_raster_data(raster):
    with rasterio.open(raster) as r:
        crs = r.crs
        bounds = r.bounds
        bbox = [bounds.left, bounds.bottom, bounds.right, bounds.top]
        footprint = Polygon([
            [bounds.left, bounds.bottom],
            [bounds.left, bounds.top],
            [bounds.right, bounds.top],
            [bounds.right, bounds.bottom]
        ])
        data = r.read(1) #TODO: wouldn't work if there are multiple bands

        # id = os.path.splitext(raster)[0]
        id = os.path.basename(raster)
        gsd = 10
        shape = r.shape
        data_type = str(r.dtypes[0])
        
        return (data,
                bbox,
                mapping(footprint),
                crs,
                id,
                gsd,
                shape,
                data_type
                )

In [119]:
raster_data,bbox,footprint,crs,id,gsd,shape,data_type = read_raster_data(raster_path)

In [120]:
raster_path

'../data/jaunpur_badlapur_2019-07-01_2020-06-30_LULCmap_10m.tif'

In [121]:
raster_item = pystac.Item(id=id,
                      geometry=footprint,
                      bbox=bbox,
                      datetime=datetime.datetime.now(datetime.timezone.utc),
                      properties={
                        #   title
                        # description
                          # "gsd": gsd, #adding this in raster extension 
                      })

Add some relevant metadata under the projection extension

In [122]:
proj_ext = ProjectionExtension.ext(raster_item, add_if_missing=True)
proj_ext.epsg = crs
proj_ext.shape = [shape[0], shape[1]]

In [123]:
raster_item

<Item id=jaunpur_badlapur_2019-07-01_2020-06-30_LULCmap_10m.tif>

In [124]:
data_url = constants.data_url

In [125]:
os.path.join(data_url, os.path.relpath(raster_path, start=data_dir))

'https://raw.githubusercontent.com/Nirzaree/STAC-spec/stac-spec-common/data/jaunpur_badlapur_2019-07-01_2020-06-30_LULCmap_10m.tif'

In [126]:
os.path.relpath(raster_path, start=data_dir)

'jaunpur_badlapur_2019-07-01_2020-06-30_LULCmap_10m.tif'

In [127]:
raster_item.add_asset("data", Asset(
    href=os.path.join(data_url, os.path.relpath(raster_path, start=data_dir)), #TODO
    media_type=MediaType.GEOTIFF,
    roles=["data"],
    title="Raster Layer"
))

In [128]:
raster_ext = RasterExtension.ext(raster_item.assets["data"], add_if_missing=True)
raster_band = RasterBand.create(
    data_type=data_type, 
    spatial_resolution=gsd,
    # nodata=nodata
)
raster_ext.bands = [raster_band]

In [129]:
def parse_raster_style_file(style_file_path):
    tree = ET.parse(style_file_path)
    root = tree.getroot()
    classes = []

    for entry in root.findall(".//paletteEntry"):
        class_info = {}
        for attr_key, attr_value in entry.attrib.items():
            if attr_key == "value":
                try:
                    class_info[attr_key] = int(attr_value)
                except ValueError:
                    class_info[attr_key] = attr_value
            else:
                class_info[attr_key] = attr_value
        classes.append(class_info)

    # If no paletteEntry tags are found, check for item tags
    if not classes:
        for entry in root.findall(".//item"):
            class_info = {}
            for attr_key, attr_value in entry.attrib.items():
                if attr_key == "value":
                    try:
                        class_info[attr_key] = int(attr_value)
                    except ValueError:
                        class_info[attr_key] = attr_value
                else:
                    class_info[attr_key] = attr_value
            classes.append(class_info)
    return classes

In [130]:
style_info = parse_raster_style_file(style_file_path=raster_style_path)

In [131]:
# style_info

In [132]:
classification_ext = ClassificationExtension.ext(raster_item.assets["data"], add_if_missing=True)
stac_classes = []
for cls in style_info:
    stac_class_obj = Classification.create(
        value=int(cls["value"]),
        name=cls.get("label") or f"Class {cls['value']}",
        description=cls.get("label"),
        color_hint=cls['color'].replace('#','')
    )
    stac_classes.append(stac_class_obj)
classification_ext.classes = stac_classes

In [133]:
# raster_item

In [134]:
raster_item.add_asset("style", Asset(
    href=os.path.join(data_url, os.path.relpath(raster_style_path, start=data_dir)),
    media_type=MediaType.XML,
    roles=["metadata"],
    title="Raster Style (QML)"
))

In [135]:
# raster_item

In [136]:
def generate_raster_thumbnail(raster_data,
                              style_info,
                              output_path
                              ):
    
    unique_raster_values = np.unique(raster_data.compressed() if isinstance(raster_data, np.ma.MaskedArray) else raster_data)
    # Filter QML info to only include values present in the raster data
    filtered_style_info = [cls for cls in style_info if cls.get('value') in unique_raster_values]
    
    values = [cls['value'] for cls in filtered_style_info if 'value' in cls]
    colors = [cls['color'] for cls in filtered_style_info if 'color' in cls]
    
    # print(f"Parsed QML values: {values}")
    # print(f"Parsed QML colors: {colors}")
        
    try:
        if not values or not colors or len(values) != len(colors):
            raise ValueError("Invalid or insufficient palette information in QML file.")
    
        sorted_indices = np.argsort(values)
        sorted_values = np.array(values)[sorted_indices]
        sorted_colors = np.array(colors)[sorted_indices]

        cmap = ListedColormap(sorted_colors)
        bounds = np.array(sorted_values) - 0.5
        bounds = np.append(bounds, sorted_values[-1] + 0.5)
        norm = Normalize(vmin=bounds.min(), vmax=bounds.max())

    except ValueError as e:
        print(f"Skipping palette generation due to error: {e}. Using a default colormap.")
        cmap = 'gray'
        norm = None
    plt.figure(figsize=(3, 3), dpi=100)
    
    plt.imshow(raster_data, cmap=cmap, norm=norm, interpolation='none')
    plt.axis('off')

    #os.makedirs(os.path.dirname(out_path), exist_ok=True)
    plt.savefig(output_path, bbox_inches='tight', pad_inches=0)
    plt.close()

In [137]:
os.makedirs(os.path.dirname(raster_thumbnail_path),exist_ok=True)

In [138]:
generate_raster_thumbnail(
    raster_data=raster_data,
    style_info=style_info,
    output_path= raster_thumbnail_path 
)

In [139]:
raster_item.add_asset("thumbnail", Asset(
    href=os.path.join(data_url, os.path.relpath(raster_thumbnail_path, start=data_dir)),
    media_type=MediaType.PNG,
    roles=["thumbnail"],
    title="Raster Thumbnail (QML)"
))

In [140]:
raster_item

<Item id=jaunpur_badlapur_2019-07-01_2020-06-30_LULCmap_10m.tif>

Vector Layer

In [141]:
def read_vector_data(vector_path,
                     target_crs='4326'
                     ):
    vector_gdf = gpd.read_file(vector_path)
    vector_gdf = vector_gdf.to_crs(epsg=target_crs)
    #TODO: remove such constants like here in crs. make it standard. available in constants. 
    bounds = vector_gdf.total_bounds
    bbox = [float(b) for b in bounds] #footprint also in vector
    geom = mapping(vector_gdf.union_all())
    
    # id = os.path.splitext(vector_path)[0]
    id = os.path.basename(vector_path)

    return (vector_gdf,bounds,bbox,geom,id)

In [142]:
vector_gdf,bounds,bbox,geom,id = read_vector_data(vector_path=vector_path)

In [143]:
vector_item = pystac.Item(
    id=id,
    geometry=geom,
    bbox=bbox,
    datetime=datetime.datetime.now(datetime.timezone.utc),
    properties={
        # "title": title,
        # "description": f"Vector data for {os.path.splitext(vector_filename)[0]} in {block} of {state_title}",
        # "start_datetime": start_date.isoformat() + 'Z',
        # "end_datetime": end_date.isoformat() + 'Z',
    }
)

In [144]:
# vector_item

Read column descriptions data

In [145]:
table_ext = TableExtension.ext(vector_item, add_if_missing=True)

In [146]:
vector_desc_filtered_df[['column_name','column_description']]

,column_name,column_description
80,no_town_cities_benefited,number of town/cities benefited from water body
81,area_21-22,area of surface water (in hectares) in 2021-2022
82,census_id,census id
83,k_19-20,percentage of water availability in kharif sea...
84,ismissing,indicates if the record has missing data
...,...,...
193,krz_20-21,"percentage of water availability in kharif, ra..."
194,krz_21-22,"percentage of water availability in kharif, ra..."
195,krz_22-23,"percentage of water availability in kharif, ra..."
196,krz_23-24,"percentage of water availability in kharif, ra..."


In [147]:
vector_merged_df = vector_gdf.dtypes.reset_index()
vector_merged_df.columns = ['column_name','column_dtype']
vector_merged_df = vector_merged_df.merge(vector_desc_filtered_df[['column_name','column_description']],
                                          on='column_name',
                                          how='left').fillna('')

In [148]:
table_ext.columns = [
    {
        "name": row['column_name'],
        "type": str(row['column_dtype']),
        "description" : row['column_description']
    }
    for ind,row in vector_merged_df.iterrows()
]


In [149]:
if vector_path.endswith('.geojson'):
    media_type = MediaType.GEOJSON

In [150]:
vector_item.add_asset("data", Asset(
    href=os.path.join(data_url, os.path.relpath(vector_path, start=data_dir)),
    media_type=media_type,
    roles=["data"],
    title="Vector Layer"
))

In [151]:
# vector_item

Generate vector thumbnail

In [152]:
def rgba_to_hex(rgba_tuple):
    if rgba_tuple is None:
        return '#808080'  # Default gray
    r, g, b, a = rgba_tuple
    return f"#{int(r*255):02x}{int(g*255):02x}{int(b*255):02x}"


In [153]:
def extract_styling_info(symbol_element):
    fill_color = None
    outline_color = None
    line_width = None

    if symbol_element is None:
        return fill_color, outline_color, line_width

    
    fill_layer = symbol_element.find('.//layer[@class="SimpleFill"]')
    if fill_layer is not None:
        color_option = fill_layer.find('Option[@name="color"]')
        if color_option is not None:
            try:
                rgb_parts = [int(p) for p in color_option.get('value').split(',')[:3]]
                fill_color = tuple([p / 255 for p in rgb_parts])
            except (ValueError, TypeError):
                fill_color = None

        outline_option = fill_layer.find('Option[@name="outline_color"]')
        if outline_option is not None:
            try:
                rgb_parts = [int(p) for p in outline_option.get('value').split(',')[:3]]
                outline_color = tuple([p / 255 for p in rgb_parts])
            except (ValueError, TypeError):
                outline_color = None
        
        width_option = fill_layer.find('Option[@name="outline_width"]')
        if width_option is not None:
            try:
                line_width = float(width_option.get('value'))
            except (ValueError, TypeError):
                line_width = None

    
    line_layer = symbol_element.find('.//layer[@class="SimpleLine"]')
    if line_layer is not None:
        color_option = line_layer.find('Option[@name="line_color"]')
        if color_option is not None:
            try:
                rgb_parts = [int(p) for p in color_option.get('value').split(',')[:3]]
                outline_color = tuple([p / 255 for p in rgb_parts])
            except (ValueError, TypeError):
                outline_color = None
        
        width_option = line_layer.find('Option[@name="line_width"]')
        if width_option is not None:
            try:
                line_width = float(width_option.get('value'))
            except (ValueError, TypeError):
                line_width = None
    
    return fill_color, outline_color, line_width

In [154]:
def parse_vector_style_file(qml_path):
    if not os.path.exists(qml_path):
        print(f"QML file not found: {qml_path}")
        return None
    
    try:
        tree = ET.parse(qml_path)
        root = tree.getroot()
        renderer_element = root.find('.//renderer-v2')

        if renderer_element is None:
            print("No renderer-v2 element found.")
            return None

        renderer_type = renderer_element.get('type')
        style = {'renderer_type': renderer_type}
        symbols = {s.get('name'): s for s in root.findall('.//symbols/symbol')}

        if renderer_type == 'singleSymbol':
            symbol_element = renderer_element.find('.//symbol') or symbols.get(renderer_element.get('symbol'))
            if symbol_element is not None:
                
                color_option = symbol_element.find('.//layer/Option[@name="line_color"]') or symbol_element.find('.//layer/Option[@name="color"]')
                
                if color_option is not None:
                    color_value = color_option.get('value').split(',')[0:3]
                    rgb_parts = [int(p) for p in color_value]
                    style['color'] = (rgb_parts[0] / 255, rgb_parts[1] / 255, rgb_parts[2] / 255)
                else:
                    
                    color_prop = symbol_element.find('.//prop[@k="color"]')
                    if color_prop is not None:
                        rgb_parts = [int(p) for p in color_prop.get('v').split(',')[:3]]
                        style['color'] = (rgb_parts[0] / 255, rgb_parts[1] / 255, rgb_parts[2] / 255)
                    else:
                        print(f"Warning: Single symbol color not found in {qml_path}.")
                        return None
            else:
                print(f"Warning: Could not find symbol element for singleSymbol in {qml_path}.")
                return None

        elif renderer_type == 'categorizedSymbol':
            style['attribute'] = renderer_element.get('attr')
            style['categories'] = []
            for cat in renderer_element.findall('categories/category'):
                symbol_element = cat.find('symbol') or symbols.get(cat.get('symbol'))
                fill_color, outline_color, line_width = extract_styling_info(symbol_element)
                style['categories'].append({
                    'value': cat.get('value'),
                    'label': cat.get('label'),
                    'fill_color': fill_color,
                    'outline_color': outline_color,
                    'line_width': line_width
                })

        elif renderer_type == 'graduatedSymbol':
            style['attribute'] = renderer_element.get('attr')
            style['classes'] = []
            for cls in renderer_element.findall('classes/class'):
                symbol_element = cls.find('symbol') or symbols.get(cls.get('symbol'))
                fill_color, outline_color, line_width = extract_styling_info(symbol_element)
                style['classes'].append({
                    'lower_bound': float(cls.get('lower')),
                    'upper_bound': float(cls.get('upper')),
                    'label': cls.get('label'),
                    'fill_color': fill_color,
                    'outline_color': outline_color,
                    'line_width': line_width
                })
        
        elif renderer_type == 'RuleRenderer':
            style['rules'] = []
            for rule in renderer_element.findall('.//rule'):
                symbol_element = rule.find('.//symbol')
                fill_color, outline_color, line_width = extract_styling_info(symbol_element)
                style['rules'].append({
                    'filter': rule.get('filter'),
                    'label': rule.get('label'),
                    'fill_color': fill_color,
                    'outline_color': outline_color,
                    'line_width': line_width
                })
        else:
            print(f"Warning: Unsupported renderer type '{renderer_type}'. Using default style.")
            return None
        return style
    except Exception as e:
        print(f"Error parsing QML file {qml_path}: {e}")
        return None

In [155]:
def generate_vector_thumbnail(vector_gdf,
                              qml_path,
                              out_path
                              ):
    
    try:
        # vector_gdf = gpd.read_file(vector_path)
        style_info = parse_vector_style_file(qml_path)

        fig, ax = plt.subplots(figsize=(6, 6))
        
        default_fill_color = (0.8, 0.8, 0.8, 1.0) # Light gray
        default_outline_color = (0, 0, 0, 1.0)   # Black
        default_line_width = 1.0

        if style_info is None:
            print("Applying default style due to parsing error.")
            vector_gdf.plot(ax=ax,
                            color=rgba_to_hex(default_fill_color), 
                            edgecolor=rgba_to_hex(default_outline_color),
                            linewidth=default_line_width)
        
        elif style_info.get('renderer_type') == 'singleSymbol':
            print("Applying single symbol style...")
            fill_color = style_info.get('fill_color', default_fill_color)
            outline_color = style_info.get('outline_color', default_outline_color)
            line_width = style_info.get('line_width', default_line_width)
            vector_gdf.plot(ax=ax,
                            color=rgba_to_hex(fill_color),
                            edgecolor=rgba_to_hex(outline_color),
                            linewidth=line_width)

        elif style_info.get('renderer_type') == 'categorizedSymbol':
            print("Applying categorized style...")
            
            color_map = {
                cat.get('value'): rgba_to_hex(cat.get('fill_color', default_fill_color))
                for cat in style_info.get('categories', [])
            }
            
            outline_color_map = {
                cat.get('value'): rgba_to_hex(cat.get('outline_color', default_outline_color))
                for cat in style_info.get('categories', [])
            }

            attribute_name = style_info.get('attribute')

            if attribute_name not in vector_gdf.columns:
                print(f"Error: Attribute column '{attribute_name}' not found. Applying default style.")
                vector_gdf.plot(ax=ax,
                                color=rgba_to_hex(default_fill_color),
                                edgecolor=rgba_to_hex(default_outline_color),
                                linewidth=default_line_width)
            else:
                vector_gdf['mapped_value'] = vector_gdf[attribute_name].apply(lambda x: str(x).strip() if pd.notnull(x) else None)
                
                fill_colors = vector_gdf['mapped_value'].map(color_map)
                fill_colors = fill_colors.fillna(rgba_to_hex(default_fill_color))

                outline_colors = vector_gdf['mapped_value'].map(outline_color_map)
                outline_colors = outline_colors.fillna(rgba_to_hex(default_outline_color))
                
                vector_gdf.plot(ax=ax, color=fill_colors, edgecolor=outline_colors, linewidth=default_line_width)
            

        elif style_info.get('renderer_type') == 'graduatedSymbol':
            print("Applying graduated style...")
            attribute_name = style_info.get('attribute')
            if attribute_name not in vector_gdf.columns:
                print(f"Error: Attribute column '{attribute_name}' not found. Applying default style.")
                vector_gdf.plot(ax=ax,
                                color=rgba_to_hex(default_fill_color),
                                edgecolor=rgba_to_hex(default_outline_color),
                                linewidth=default_line_width)
            else:
                fill_colors = []
                for _, row in vector_gdf.iterrows():
                    val = row[attribute_name]
                    found_color = default_fill_color
                    for cls in style_info.get('classes', []):
                        if cls.get('lower_bound') is not None and cls.get('upper_bound') is not None:
                            if cls['lower_bound'] <= val < cls['upper_bound']:
                                found_color = cls.get('fill_color', default_fill_color)
                                break
                    fill_colors.append(rgba_to_hex(found_color))
                
                vector_gdf.plot(ax=ax,
                                color=fill_colors,
                                edgecolor=rgba_to_hex(default_outline_color),
                                linewidth=default_line_width)

        elif style_info.get('renderer_type') == 'RuleRenderer':
            print("Applying rule-based style...")
            fill_colors = []
            for _, row in vector_gdf.iterrows():
                assigned_color = default_fill_color
                for rule in style_info.get('rules', []):
                    try:
                        attribute_name = rule['filter'].split(' ')[0].strip().strip('"').strip("'")
                        if attribute_name in row and pd.eval(rule['filter'], local_dict={attribute_name: row[attribute_name]}):
                            assigned_color = rule.get('fill_color', default_fill_color)
                            break
                    except Exception:
                        continue 
                fill_colors.append(rgba_to_hex(assigned_color))

            vector_gdf.plot(ax=ax,
                            color=fill_colors,
                            edgecolor=rgba_to_hex(default_outline_color),
                            linewidth=default_line_width)

        else:
            print("Applying default blue style.")
            vector_gdf.plot(ax=ax,
                            color='lightblue',
                            edgecolor=rgba_to_hex(default_outline_color),
                            linewidth=default_line_width)

        ax.set_axis_off()
        plt.tight_layout()
        plt.savefig(out_path)
        plt.close(fig)
        print(f"Thumbnail saved to: {out_path}")

    except Exception as e:
        print(f"Error generating vector thumbnail: {e}")

In [156]:
generate_vector_thumbnail(vector_gdf = vector_gdf,
                          qml_path = vector_style_path,
                          out_path = vector_thumbnail_path
                          )

Applying default style due to parsing error.
Thumbnail saved to: ../data/STAC_output/swb3_jaunpur_badlapur_thumbnail.png


In [157]:
vector_item.add_asset("thumbnail", Asset(
    href=os.path.join(data_url, os.path.relpath(vector_thumbnail_path, start=data_dir)),
    media_type=MediaType.PNG,
    roles=["thumbnail"],
    title="Vector Thumbnail"
))

In [158]:
vector_item.add_asset("style", Asset(
    href=os.path.join(data_url, os.path.relpath(vector_style_path, start=data_dir)),
    media_type=MediaType.XML,
    roles=["metadata"],
    title="Vector Style (QML)"
))

In [159]:
#Getting actual resolution without exporting the data from ee 

# LULC = ee.Image('projects/ee-corestackdev/assets/apps/mws/uttar_pradesh/jaunpur/badlapur/jaunpur_badlapur_2017-07-01_2018-06-30_LULCmap_10m')
# LULC.projection().nominalScale().getInfo()

Create block catalog

In [160]:
block = 'badlapur'

In [161]:
district = block_district_state_df[block_district_state_df['block'] == block]['district'].iloc[0]
state = block_district_state_df[block_district_state_df['block'] == block]['state'].iloc[0]
print(state,district,block)

uttar_pradesh jaunpur badlapur


In [162]:
block_catalog = pystac.Catalog(
    id=block,
    title=f"{block}",
    description=f"STAC catalog for {block} block data in {district}, {state}"
)

In [163]:
block_dir = os.path.join(corestack_dir, state, district, block)
os.makedirs(block_dir, exist_ok=True)

In [164]:
block_catalog.add_item(raster_item)

<Link rel=item target=<Item id=jaunpur_badlapur_2019-07-01_2020-06-30_LULCmap_10m.tif>>

In [165]:
block_catalog.add_item(vector_item)

<Link rel=item target=<Item id=swb3_jaunpur_badlapur.geojson>>

In [166]:
block_catalog.normalize_and_save(block_dir,
                                 catalog_type=pystac.CatalogType.SELF_CONTAINED)

Add/update district,state, and root catalogs/collections

In [167]:
root_catalog_path = os.path.join(corestack_dir, "catalog.json")
root_catalog_path

'../data/CorestackCatalogs/catalog.json'

In [168]:
root_catalog_path = os.path.join(corestack_dir, "catalog.json")

if os.path.exists(root_catalog_path):
    root_catalog = pystac.read_file(root_catalog_path)
    print("Loaded existing root catalog.")
else:
    os.makedirs(corestack_dir, exist_ok=True)
    root_catalog = pystac.Catalog(
        id="corestack_STAC",
        title=constants.root_catalog_title,
        description=constants.root_catalog_description
    )
    root_catalog.set_self_href(root_catalog_path)
    print("Created new root catalog.")

Created new root catalog.


Read root catalog 

In [169]:
# pystac.Catalog.from_file('https://raw.githubusercontent.com/Nirzaree/STAC-spec/stac-spec-common/data/CorestackCatalogs/catalog.json')

In [170]:
state

'uttar_pradesh'

In [171]:
state_dir = os.path.join(corestack_dir, state)
state_collection_path = os.path.join(state_dir, "collection.json")
print(state_collection_path)

../data/CorestackCatalogs/uttar_pradesh/collection.json


In [172]:
state_collections = {}
district_catalogs = {}

In [173]:
if os.path.exists(state_collection_path):
    state_collection = pystac.read_file(state_collection_path)
    print(f"Loaded existing state collection: {state}")
else:
    os.makedirs(state_dir, exist_ok=True)
    state_collection = pystac.Collection(
        id=state,
        title= f"{state}", #f"{state_title}",
        description=f"STAC Collection for data of {state} state.",
        extent=pystac.Extent(
            spatial=pystac.SpatialExtent([0, 0, 0, 0]),
            temporal=pystac.TemporalExtent(
                [[constants.DEFAULT_START_DATE, constants.DEFAULT_END_DATE]])
        ),
        license="https://spdx.org/licenses/CC-BY-4.0.html",
        providers=[
            pystac.Provider(
                name="Corestack",
                roles=[pystac.ProviderRole.PRODUCER,
                       pystac.ProviderRole.PROCESSOR],
                url="https://core-stack.org/"
            )
        ],
        keywords=["geospatial", "remote sensing", block, district, state],
    )
    
    state_collection.add_link(
        pystac.Link(
            rel="documentation",
            target="https://core-stack.org/",
            title="CoRE stack",
            media_type="application/pdf"
        )
    )
    state_collection.add_link(
        pystac.Link(
            rel="documentation",
            target="https://drive.google.com/file/d/1ZxovdpPThkN09cB1TcUYSE2BImI7M3k_/view",
            title="Technical Manual",
            media_type="application/pdf"
        )
    )
    
    state_collection.add_link(
        pystac.Link(
            rel="documentation",
            target="https://github.com/Nirzaree/STAC-spec/tree/stac-spec-common",
            title="Github link",
            media_type="application/pdf"
        )
    )
    


    
    state_collection.normalize_and_save(
        state_dir, catalog_type=pystac.CatalogType.SELF_CONTAINED)
    print(f"Created new state collection: {state}")

Created new state collection: uttar_pradesh


In [174]:
root_catalog.add_child(state_collection)
print(f"State collection '{state}' linked in root catalog.")
state_collections[state] = state_collection    

State collection 'uttar_pradesh' linked in root catalog.


In [175]:
if (state, district) not in district_catalogs:
    district_dir = os.path.join(corestack_dir, state, district)
    district_catalog_path = os.path.join(district_dir, "catalog.json")
    if os.path.exists(district_catalog_path):
        district_catalog = pystac.read_file(district_catalog_path)
        print(f"Loaded existing district catalog: {district}")
    else:
        os.makedirs(district_dir, exist_ok=True)
        district_catalog = pystac.Catalog(
            id=district,
            title= f"{district}",#f"{district_title}",
            description=f"STAC catalog for data of {district} district"
        )
        district_catalog.normalize_and_save(district_dir, catalog_type=pystac.CatalogType.SELF_CONTAINED)
        print(f"Created new district catalog: {district}")
    
    state_collections[state].add_child(district_catalog)
    print(f"Added district catalog '{district}' to state collection '{state}'.")
    district_catalogs[(state, district)] = district_catalog

Created new district catalog: jaunpur
Added district catalog 'jaunpur' to state collection 'uttar_pradesh'.


In [176]:
block_catalog_dir = os.path.join(corestack_dir, state, district, block)
block_catalog = pystac.read_file(os.path.join(block_catalog_dir, 'catalog.json'))

existing_child_ids_in_district = {child.id for child in district_catalogs[(state, district)].get_children()}

if block not in existing_child_ids_in_district:
    district_catalogs[(state, district)].add_child(block_catalog)
    print(f"Added block catalog '{block}' to district catalog '{district}'.")
else:
    print(f"Block '{block}' already exists in district catalog '{district}'.")

root_catalog.normalize_and_save(corestack_dir, catalog_type=pystac.CatalogType.SELF_CONTAINED)

Added block catalog 'badlapur' to district catalog 'jaunpur'.
